# Loan Default Prediction — Exploratory Data Analysis (EDA)

## 1. Project & Objective
This notebook explores `loan_train.csv` to analyze applicant and loan characteristics that influence loan default probability, discover data patterns, inspect class imbalance, and prepare clean features for machine learning modeling without data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/loan_train.csv')
print(f'Shape of dataset: {df.shape}')
df.head()

## 3. Data Cleaning & Type Conversion
- Parse percentage columns (`int_rate`, `revol_util`)
- Parse loan `term` to integer months
- Parse `emp_length` to numeric years

In [ ]:
def clean_percentage(val):
    if pd.isna(val): return np.nan
    return float(str(val).replace('%', '').strip())

df['int_rate'] = df['int_rate'].apply(clean_percentage)
if 'revol_util' in df.columns:
    df['revol_util'] = df['revol_util'].apply(clean_percentage)

df['term'] = df['term'].str.extract('(\d+)').astype(float)
print('Cleaned percentage and term columns.')

## 4. Target Variable Analysis (`loan_status`)
- `0`: Non-default (Fully Paid)
- `1`: Default (Charged Off)

In [ ]:
target_counts = df['loan_status'].value_counts()
default_rate = df['loan_status'].mean() * 100
print(f'Target distribution:\n{target_counts}')
print(f'Default Rate: {default_rate:.2f}%')

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette=['#10b981', '#ef4444'])
plt.title('Target Distribution: Non-Default (0) vs Default (1)')
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.show()

## 5. Default Rate by Credit Grade & Term

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

grade_order = sorted([g for g in df['grade'].dropna().unique()])
grade_def = df.groupby('grade')['loan_status'].mean().reindex(grade_order) * 100
sns.barplot(x=grade_def.index, y=grade_def.values, ax=axes[0], palette='magma')
axes[0].set_title('Default Rate (%) by Credit Grade')
axes[0].set_ylabel('Default Rate (%)')

term_def = df.groupby('term')['loan_status'].mean() * 100
sns.barplot(x=term_def.index.astype(int), y=term_def.values, ax=axes[1], palette='crest')
axes[1].set_title('Default Rate (%) by Loan Term (Months)')
axes[1].set_ylabel('Default Rate (%)')

plt.tight_layout()
plt.show()

## 6. Numerical Features: Income, Interest Rate, DTI, & Loan Amount

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='loan_status', y='int_rate', data=df, ax=axes[0, 0], palette=['#10b981', '#ef4444'])
axes[0, 0].set_title('Interest Rate vs Loan Status')

sns.boxplot(x='loan_status', y='dti', data=df, ax=axes[0, 1], palette=['#10b981', '#ef4444'])
axes[0, 1].set_title('DTI vs Loan Status')

sns.boxplot(x='loan_status', y='loan_amnt', data=df, ax=axes[1, 0], palette=['#10b981', '#ef4444'])
axes[1, 0].set_title('Loan Amount vs Loan Status')

sns.boxplot(x='loan_status', y='annual_inc', data=df[df['annual_inc'] < 200000], ax=axes[1, 1], palette=['#10b981', '#ef4444'])
axes[1, 1].set_title('Annual Income (<$200k) vs Loan Status')

plt.tight_layout()
plt.show()

## 7. Numerical Correlation Matrix Heatmap

In [ ]:
num_cols = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'revol_bal', 'loan_status']
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix Heatmap')
plt.show()

## 8. Summary of Insights
- 60-month loans exhibit higher default rates than 36-month loans.
- Default risk scales continuously from grade A towards grade G.
- Borrowers with higher interest rates, higher DTI, and lower annual income carry greater default probability.
- Strict leakage prevention is applied to avoid post-issuance fields during ML model training.